<a href="https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold3_of3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlphaFold 3 (OpenFold3 Preview 2 weights)

Predict protein, RNA, DNA, and small-molecule structures using [AlphaFold 3](https://www.nature.com/articles/s41586-024-07487-w) with the freely available [OpenFold3](https://github.com/aqlaboratory/openfold) weights (Apache 2.0 — no commercial restrictions).

MSA generation via the [ColabFold](https://github.com/sokrypton/ColabFold) MMseqs2 server — **no local databases required**.

Just fill in the **Input sequence(s)** cell and run all. Proteins, nucleic acids and ligands all go in a single box; attention/XLA flags are chosen automatically for your runtime (T4, newer GPUs, or CPU).

**Citations:**
- Abramson et al. (2024) AlphaFold 3. *Nature* [doi:10.1038/s41586-024-07487-w](https://doi.org/10.1038/s41586-024-07487-w)
- Mirdita et al. (2022) ColabFold. *Nature Methods* [doi:10.1038/s41592-022-01488-1](https://doi.org/10.1038/s41592-022-01488-1)
- The OpenFold3 Team (2025) [Github](https://github.com/aqlaboratory/openfold-3) - [doi:10.5281/zenodo.19001000](https://doi.org/10.5281/zenodo.19001000)

**Credits:** AF3 code: Google DeepMind (Apache 2.0) · OF3 weights: AlQuraishi Lab (Apache 2.0)

In [ ]:
#@title Install dependencies (~3 mins)
%%time
import os, time, glob

weights_url = '' #@param {type:"string"}
#@markdown - **weights_url**: leave EMPTY to use the free, open **OpenFold3** weights. To use
#@markdown   the official **AlphaFold 3** weights, paste the Google Cloud URL: `https://storage.googleapis.com/alphafold3/models/[???]/af3.bin.zst` from your DeepMind
#@markdown   approval email (you'll be prompted to authenticate). Subject to the AF3 weights terms of use.
weights_url = weights_url.strip()
USE_NATIVE = bool(weights_url)
NATIVE_DIR = 'af3_native_weights'

if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  os.system("pip install -q 'jax[cuda12]==0.10.1' dm-haiku==0.0.16 rdkit==2025.9.4 \
  zstandard awscli tokamax==0.0.11 py3Dmol py2Dmol")
  os.system("pip install -q --no-deps 'https://github.com/sokrypton/alphafold3/releases/download/v3.1.0/alphafold3_open-3.1.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'")
  base = 'https://raw.githubusercontent.com/sokrypton/alphafold3/refs/heads/main'
  for script in ['run_alphafold', 'convert_of3_weights']:
    os.system(f'wget -q -nc {base}/{script}.py')
  os.system('touch ALPHAFOLD3_READY')
  print('Packages installed.')

# Patch tokamax so Ada/consumer GPUs (L4, A10, RTX 30/40; cc 8.6/8.9) fall back to XLA
# kernels. tokamax enables its Triton kernels for ALL cc>=8.0 GPUs, but those kernels
# need more shared memory than Ada cards have -> 'Shared memory size limit exceeded' at
# launch (which its trace-time fallback can't catch). Restrict Triton to true datacenter
# GPUs (A100 cc 8.0, H100 cc 9.0+); everything else uses XLA, exactly like the T4 path.
try:
  import tokamax
  _gu = os.path.join(os.path.dirname(tokamax.__file__), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 (8.6/8.9) lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights: official AlphaFold 3 (if a URL was given) or OpenFold3 (default, free)
if USE_NATIVE:
  if not os.path.isfile('NATIVE_WEIGHTS_DONE'):
    print('Fetching official AlphaFold 3 weights (Google authentication required)...')
    from google.colab import auth
    auth.authenticate_user()
    from google.cloud import storage
    weights_path = weights_url.removeprefix('https://storage.googleapis.com/alphafold3/')
    blob = storage.Client().bucket('alphafold3').get_blob(weights_path)
    if blob is None:
      raise FileNotFoundError(f'No weights found at {weights_url} - check the URL and that your account was granted access.')
    af3_weights = blob.open('rb').read()
    print(f'Loaded AlphaFold 3 weights ({len(af3_weights):,} bytes)')
    os.makedirs(NATIVE_DIR, exist_ok=True)
    for _f in glob.glob(f'{NATIVE_DIR}/*'):       # keep exactly one model file in the dir
      os.remove(_f)
    _name = os.path.basename(weights_path)
    if not (_name.endswith('.bin') or _name.endswith('.bin.zst')):
      _name = 'af3.bin.zst' if af3_weights[:4] == b'\x28\xb5\x2f\xfd' else 'af3.bin'  # zstd magic
    with open(f'{NATIVE_DIR}/{_name}', 'wb') as _fh:
      _fh.write(af3_weights)
    open('NATIVE_WEIGHTS_DONE', 'w').close()
    print(f'Saved native weights -> {NATIVE_DIR}/{_name}')
elif not os.path.isfile('WEIGHTS_DONE'):
  print('Downloading OpenFold3 weights...')
  os.system('(aws s3 cp s3://openfold/staging/of3-p2-155k.pt . --no-sign-request; touch WEIGHTS_DONE) &')

# Build AF3 data files (background, independent of weights)
if not os.path.isfile('DATA_DONE'):
  print('Building AF3 data files...')
  os.system('(build_data; touch DATA_DONE) &')

# Wait for background jobs (the OF3 download only matters for the OpenFold3 path)
for sentinel in (['DATA_DONE'] if USE_NATIVE else ['WEIGHTS_DONE', 'DATA_DONE']):
  while not os.path.isfile(sentinel):
    time.sleep(5)
  print(f'{sentinel} ✓')

if not USE_NATIVE and not os.path.isfile('af3_converted_weights/of3_ported_weights.bin.zst'):
  print("converting openfold3 weights...")
  os.system("python convert_of3_weights.py --of3_checkpoint of3-p2-155k.pt --output_dir af3_converted_weights")

print('Setup complete!  ' + ('Using official AlphaFold 3 weights.' if USE_NATIVE else 'Using OpenFold3 weights.'))

In [ ]:
#@title Input sequences
import re, os, json, hashlib

#@markdown ### Molecules
#@markdown Separate multiple chains within a box using `:` (extra colons are fine: `A::::B` == `A:B`). Leave a box empty if unused; full details in the Instructions cell.
protein = 'PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK' #@param {type:"string"}
dna = '' #@param {type:"string"}
rna = '' #@param {type:"string"}
ligand_ccd = '' #@param {type:"string"}
ligand_smiles = '' #@param {type:"string"}

#@markdown ### Run settings
jobname = 'test' #@param {type:"string"}
msa_mode = "mmseqs2_server" #@param ["mmseqs2_server", "single_sequence"]
seeds = '1' #@param {type:"string"}
on_existing = "overwrite" #@param ["overwrite", "skip"]
#@markdown - `msa_mode`: `single_sequence` skips the MSA (faster, lower accuracy).
#@markdown - `seeds`: comma-separated, e.g. `1,2,3`.
#@markdown - `on_existing`: `overwrite` replaces this job's previous results; `skip` keeps them.

# Split a box into entries: collapse colon runs, drop whitespace, skip empties
def split_entries(s):
  s = re.sub(r':+', ':', s).strip(':')
  return [e for e in (''.join(tok.split()) for tok in s.split(':')) if e]

prot_seqs   = [e.upper() for e in split_entries(protein)]
dna_seqs    = [e.upper() for e in split_entries(dna)]
rna_seqs    = [e.upper() for e in split_entries(rna)]
ccd_codes   = [e.upper() for e in split_entries(ligand_ccd)]
smiles_strs = split_entries(ligand_smiles)         # case-sensitive: leave as typed

# Build AF3 chain entities (IDs A, B, C, ... in canonical order)
CHAIN_IDS = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz')
chains, prot_groups, idx = [], {}, 0

for seq in prot_seqs:
  cid = CHAIN_IDS[idx]; idx += 1
  if seq in prot_groups:                           # merge identical seqs -> homo-oligomer
    ent = prot_groups[seq]
    ids = ent['id'] if isinstance(ent['id'], list) else [ent['id']]
    ent['id'] = ids + [cid]
  else:
    ent = {'id': cid, 'sequence': seq, 'templates': []}
    if msa_mode == 'single_sequence':
      ent.update({'unpairedMsa': f'>query\n{seq}\n', 'pairedMsa': ''})
    prot_groups[seq] = ent
    chains.append({'protein': ent})

for seq in rna_seqs:
  c = {'id': CHAIN_IDS[idx], 'sequence': seq}
  if msa_mode == 'single_sequence':
    c['unpairedMsa'] = f'>query\n{seq}\n'
  chains.append({'rna': c}); idx += 1

for seq in dna_seqs:
  chains.append({'dna': {'id': CHAIN_IDS[idx], 'sequence': seq}}); idx += 1

for code in ccd_codes:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'ccdCodes': [code]}}); idx += 1

for smiles in smiles_strs:
  chains.append({'ligand': {'id': CHAIN_IDS[idx], 'smiles': smiles}}); idx += 1

if not chains:
  raise ValueError('No valid input found - fill in at least one box.')

# Seeds: pull out integers regardless of separators, dedupe, default to [1]
seed_list = []
for tok in re.findall(r'\d+', seeds):
  v = int(tok)
  if v not in seed_list:
    seed_list.append(v)
if not seed_list:
  seed_list = [1]

# Deterministic, lower-cased job name from inputs+seeds.
# Same input+seeds -> same folder (so re-runs reuse it instead of piling up).
# Lower-cased to match run_alphafold.py's sanitised_name() output directory.
def _flat(mol):
  if 'sequence' in mol: return mol['sequence']
  if 'ccdCodes' in mol: return ','.join(mol['ccdCodes'])
  return mol.get('smiles', '?')
flat = ':'.join(_flat(list(c.values())[0]) for c in chains) + '|seeds=' + ','.join(map(str, seed_list))
basejob = (re.sub(r'\W+', '', ''.join(jobname.split())) or 'job').lower()
jobname = basejob + '_' + hashlib.sha1(flat.encode()).hexdigest()[:5]

# Input JSON goes to a temp dir; ALL results land in ONE folder: af3_output/<jobname>/
INPUT_DIR  = '/tmp/af3_inputs'
OUTPUT_DIR = 'af3_output'
job_dir    = f'{OUTPUT_DIR}/{jobname}'

fold_input = {
    'name': jobname,
    'sequences': chains,
    'modelSeeds': seed_list,
    'dialect': 'alphafold3',
    'version': 1,
}
os.makedirs(INPUT_DIR, exist_ok=True)
json_path = f'{INPUT_DIR}/{jobname}.json'
with open(json_path, 'w') as f:
  json.dump(fold_input, f, indent=2)

print(f'Job "{jobname}"  ->  results will be written to {job_dir}/')
fold_input


In [ ]:
#@title Run AlphaFold 3
%%time
import os, shutil, subprocess, glob

#@markdown Inference settings (defaults match AlphaFold 3 - increase only if needed):
num_recycles = 10 #@param {type:"integer"}
num_diffusion_samples = 5 #@param {type:"integer"}
#@markdown - `num_recycles`: refinement passes through the network (default 10). More can help large/hard targets, but is slower.
#@markdown - `num_diffusion_samples`: candidate structures generated per seed (default 5). Total models = seeds x samples.

num_recycles = max(1, int(num_recycles))
num_diffusion_samples = max(1, int(num_diffusion_samples))

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Re-run policy (one folder per job, no timestamped duplicates):
#   overwrite -> wipe this job's folder and recompute
#   skip      -> if a finished result (.cif) is already there, don't recompute
have_results = os.path.isdir(job_dir) and any(f.endswith('.cif') for f in os.listdir(job_dir))
run_it = not (on_existing == 'skip' and have_results)
if run_it:
  shutil.rmtree(job_dir, ignore_errors=True)   # start clean so exactly one folder is produced

# Pick attention impl + XLA flags from the actual device.
# Triton/cuDNN flash attention need Ampere (compute capability >= 8.0);
# 7.x GPUs (T4=7.5, V100=7.0) and CPU use the portable XLA path, and 7.x
# additionally needs the XLA flag that disables the custom-kernel fusion pass.
def detect_device():
  try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15)
    caps = [float(x) for x in out.stdout.split() if x.strip()]
    if caps:
      return 'gpu', min(caps)
  except Exception:
    pass
  return 'cpu', None

device, cap = detect_device()
nojit = False
xla_flags = []   # extra XLA flags to export for this device (per AlphaFold 3's guidance)

if device == 'cpu':
  flash_impl = 'xla'
  nojit = True
  print('No GPU detected - running on CPU with XLA attention + --nojit (slow, but avoids the compile).')
elif cap < 8.0:
  # T4 / V100 (cc 7.x): XLA attention; disable the custom-kernel fusion pass.
  # (Triton GEMM is not supported on these cards, so it is not disabled here.)
  flash_impl = 'xla'
  xla_flags = ['--xla_disable_hlo_passes=custom-kernel-fusion-rewriter']
  print(f'Pre-Ampere GPU (compute capability {cap}) - XLA attention + custom-kernel fusion disabled.')
elif 8.0 < cap < 9.0:
  # L4 / Ada / consumer Ampere (cc 8.6 / 8.9): limited shared memory. XLA's Triton GEMM
  # kernels exceed it ('Shared memory size limit exceeded'), so disable Triton GEMM
  # (falls back to cuBLAS) and use XLA attention to also avoid the Triton attention kernel.
  flash_impl = 'xla'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Ada/consumer GPU (compute capability {cap}) - XLA attention + Triton GEMM disabled (shared-memory limit).')
else:
  # A100 (cc 8.0) and H100 (cc 9.0+): ample shared memory. Triton flash attention,
  # with Triton GEMM disabled per AlphaFold 3's recommended XLA_FLAGS.
  flash_impl = 'triton'
  xla_flags = ['--xla_gpu_enable_triton_gemm=false']
  print(f'Datacenter GPU (compute capability {cap}) - Triton flash attention + Triton GEMM disabled.')

# Export XLA flags so the child shell (and JAX inside it) inherit them.
cur = os.environ.get('XLA_FLAGS', '')
for f in xla_flags:
  if f not in cur:
    cur = (cur + ' ' + f).strip()
if cur:
  os.environ['XLA_FLAGS'] = cur

print('XLA_FLAGS =', os.environ.get('XLA_FLAGS', '(unset)'))

# Weights: use the official AlphaFold 3 weights if they were fetched, else OpenFold3.
if os.path.isdir('af3_native_weights') and glob.glob('af3_native_weights/*.bin*'):
  model_dir, use_of3 = 'af3_native_weights', False
  print('Weights: official AlphaFold 3')
else:
  model_dir, use_of3 = 'af3_converted_weights', True
  print('Weights: OpenFold3')

cmd = [
    'python', 'run_alphafold.py',
    f'--json_path={json_path}',
    f'--model_dir={model_dir}',
    '--norun_data_pipeline',
    f'--output_dir={OUTPUT_DIR}',
    '--cache_dir=/tmp/af3_cache',
    '--force_output_dir',          # reuse af3_output/<jobname>/ instead of a timestamped copy
    f'--flash_attention_implementation={flash_impl}',
    f'--num_recycles={num_recycles}',
    f'--num_diffusion_samples={num_diffusion_samples}',
]
if msa_mode == 'mmseqs2_server':
  cmd.append('--use_msa_server')
if nojit:
  cmd.append('--nojit')
if use_of3:
  cmd.append('--of3_weights')

cmd = ' '.join(cmd)
if run_it:
  print(cmd)
  !{cmd}
  print(f'\nDone -> {job_dir}/')
else:
  print(f'Skipping: results already exist in {job_dir}/  (set on_existing=overwrite to recompute).')


In [ ]:
#@title Display structures + PAE (py2Dmol)
import csv, glob, os, json
import numpy as np
import py2Dmol

load_as_frames = True #@param {type:"boolean"}
viewer_size = 400
#@markdown All predicted models load together, best first (**rank_1, rank_2, ...**), each with its own PAE.
#@markdown - `load_as_frames` **off** -> pick a model from the dropdown.
#@markdown - `load_as_frames` **on**  -> models become frames you can play through (press play / drag the slider).
#@markdown - The interactive PAE matrix sits beside the structure; click or drag-box on it to highlight residues.

# All models in rank order (best first): from the ranking CSV, fall back to globbing.
def collect_models():
  ranking_csv = f'{job_dir}/{jobname}_ranking_scores.csv'
  cifs = []
  if os.path.exists(ranking_csv):
    rows = []
    with open(ranking_csv) as f:
      for r in csv.DictReader(f):
        rows.append((float(r['ranking_score']), int(r['seed']), int(r['sample'])))
    for _, seed, sample in sorted(rows, reverse=True):
      d = f'{job_dir}/seed-{seed}_sample-{sample}'
      hit = sorted(glob.glob(f'{d}/*_model.cif')) or sorted(glob.glob(f'{d}/*.cif'))
      if hit:
        cifs.append(hit[0])
  if not cifs:
    cifs = (sorted(glob.glob(f'{job_dir}/**/*_model.cif', recursive=True))
            or sorted(glob.glob(f'{job_dir}/**/*.cif', recursive=True)))
  return cifs

# Per-model PAE: confidences.json next to the CIF, else the top-level one.
def load_pae(cif):
  d = os.path.dirname(cif)
  cands = [p for p in glob.glob(f'{d}/*_confidences.json')
           if 'summary' not in os.path.basename(p)]
  if not cands:
    top = f'{job_dir}/{jobname}_confidences.json'
    cands = [top] if os.path.exists(top) else []
  if cands:
    pae = json.load(open(cands[0])).get('pae')
    if pae is not None:
      return np.asarray(pae, dtype=float)
  return None

cifs = collect_models()
if not cifs:
  raise FileNotFoundError(f'No model CIFs found in {job_dir}/')
print(f'Loaded {len(cifs)} model(s) from {job_dir}/'
      + ('  (frames - press play)' if load_as_frames else '  (use the dropdown to switch)'))

viewer = py2Dmol.view(size=(viewer_size, viewer_size),
                      pae=True, autoplay=load_as_frames)
for i, cif in enumerate(cifs, start=1):
  pae = load_pae(cif)
  if load_as_frames:
    viewer.add_pdb(cif, name='models', paes=pae)      # same name -> frames (play through)
  else:
    viewer.add_pdb(cif, name=f'rank_{i}', paes=pae)    # distinct names -> dropdown of objects
viewer.show()


In [ ]:
#@title Quality metrics and plots
import json, os
import numpy as np
import matplotlib.pyplot as plt

conf_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_confidences.json'
summ_path = f'{OUTPUT_DIR}/{jobname}/{jobname}_summary_confidences.json'

with open(conf_path) as f:
  conf = json.load(f)
with open(summ_path) as f:
  summ = json.load(f)

plddts = np.array(conf.get('atom_plddts', conf.get('token_plddts', [])), dtype=float)
plddt_chain_ids = conf.get('atom_chain_ids', conf.get('token_chain_ids', []))   # pLDDT is per-ATOM
token_chain_ids = conf.get('token_chain_ids', [])                                # PAE is per-TOKEN
pae = np.array(conf.get('pae', []), dtype=float)

# ── Summary (ipTM is None for single-chain jobs — guard before formatting) ─
def fmt(v):
  return f'{v:.3f}' if isinstance(v, (int, float)) else 'n/a'

mean_plddt = summ.get('mean_plddt')
if mean_plddt is None and plddts.size:
  mean_plddt = float(np.mean(plddts))
iptm = summ.get('iptm')

print('=' * 38)
print(f'Mean pLDDT     : {fmt(mean_plddt)}')
print(f'pTM            : {fmt(summ.get("ptm"))}')
print(f'ipTM           : {fmt(iptm)}' + ('   (single chain — no interface)' if iptm is None else ''))
print(f'Ranking score  : {fmt(summ.get("ranking_score"))}')
print('=' * 38)

# ── Plots ───────────────────────────────────────────────────
has_pae = pae.ndim == 2 and pae.size > 0
ncols = 2 if has_pae else 1
fig, axes = plt.subplots(1, ncols, figsize=(13 if has_pae else 6.5, 4))
axes = np.atleast_1d(axes)

# pLDDT per residue — a line (coloured per chain when there is more than one)
ax = axes[0]
x = np.arange(len(plddts))
xmax = max(len(plddts) - 1, 1)
ax.set_xlim(0, xmax)
ax.set_ylim(0, 100)

unique_chains = list(dict.fromkeys(plddt_chain_ids))
if len(unique_chains) > 1:
  colors = plt.cm.tab10(np.linspace(0, 1, len(unique_chains)))
  tcid = np.array(plddt_chain_ids)
  for ch, col in zip(unique_chains, colors):
    y = np.where(tcid == ch, plddts, np.nan)   # NaN gaps keep chains as separate lines
    ax.plot(x, y, lw=1.5, color=col, label=f'Chain {ch}')
  for b in [i for i in range(1, len(plddt_chain_ids)) if plddt_chain_ids[i] != plddt_chain_ids[i-1]]:
    ax.axvline(b - 0.5, color='grey', lw=0.6, alpha=0.5)
  ax.legend(loc='lower right', fontsize=8)
else:
  ax.plot(x, plddts, lw=1.5, color='#1f77b4')

for y in (50, 70, 90):
  ax.axhline(y, ls='--', lw=0.7, color='grey', alpha=0.5)
  ax.text(xmax, y, f' {y}', va='center', ha='left', fontsize=7, color='grey')
ax.set_xlabel('Atom')
ax.set_ylabel('pLDDT')
ax.set_title('Predicted pLDDT per atom')

# PAE matrix
if has_pae:
  ax = axes[1]
  im = ax.imshow(pae, cmap='bwr', vmin=0, vmax=30, interpolation='nearest')
  plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='PAE (Å)')
  if token_chain_ids:
    for b in [i for i in range(1, len(token_chain_ids)) if token_chain_ids[i] != token_chain_ids[i-1]]:
      ax.axhline(b - 0.5, c='black', lw=0.8)
      ax.axvline(b - 0.5, c='black', lw=0.8)
  ax.set_xlabel('Scored residue')
  ax.set_ylabel('Aligned residue')
  ax.set_title('Predicted Aligned Error (PAE)')

plt.tight_layout()
plt.show()


In [ ]:
#@title Download results
from google.colab import files
import os

results_zip = f'{jobname}.result.zip'
os.system(f'zip -r {results_zip} {OUTPUT_DIR}/{jobname}')
files.download(results_zip)


# Instructions <a name="Instructions"></a>

**Quick start**
1. Fill in the sequence(s) in the **Input sequence(s)** cell.
2. Press **Runtime → Run all**.
3. The install cell (first run only) downloads OpenFold3 weights and builds AF3 data files in the background — subsequent runs reuse them.

---

## Model weights

By default the notebook uses the open **OpenFold3** weights (Apache-2.0, no restrictions). To run the **official AlphaFold 3** weights instead, paste the Google Cloud URL from your DeepMind approval email into the **weights_url** field of the install cell. You'll be prompted to authenticate with the Google account that was granted access; the weights are fetched into `af3_native_weights/` and used automatically (the run cell detects them and drops the `--of3_weights` flag). Leaving **weights_url** empty keeps the OpenFold3 path.

Official AlphaFold 3 parameters are subject to the [AF3 weights terms of use](https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md); run_alphafold prints a reminder of these terms when they are used.

---

## Sequence input

Each molecule type has its own box. Within a box, separate multiple chains with `:`.

| Box | What goes in it | Example |
|---|---|---|
| **protein** | amino-acid sequence(s) | `MKTAY...` or `SEQ1:SEQ2` |
| **dna** | DNA sequence(s) | `CGCGAATTCGCG` |
| **rna** | RNA sequence(s) | `GCGGAUUUA` |
| **ligand_ccd** | ligand(s) by PDB CCD code | `ATP:MG:HEM` |
| **ligand_smiles** | ligand(s) by SMILES | `CC(=O)Oc1ccccc1C(=O)O` |

Chains are assigned IDs A, B, C, … following AlphaFold 3's canonical order (protein → RNA → DNA → ligand; CCD ligands before SMILES ligands). Mix freely across boxes to build a complex — e.g. a protein in **protein**, `AUGCAUGC` in **rna**, and `ATP` in **ligand_ccd**.

- **Homo-oligomers**: identical protein sequences are merged automatically, so `SEQ:SEQ` = homodimer, `SEQ:SEQ:SEQ` = homotrimer.
- Protein / DNA / RNA sequences and CCD codes are upper-cased automatically; **SMILES are left exactly as typed** (case is meaningful in SMILES).
- Spaces and newlines inside an entry are ignored, and **extra colons are forgiven** — `SEQ1::::SEQ2` is the same as `SEQ1:SEQ2`. Leave a box empty if unused.
- *Note:* because `:` separates entries, an atom-mapped SMILES that itself contains a colon (e.g. `[C:1]`) isn't supported via the box — use a raw AF3 JSON for that edge case.

## Seeds

Enter one or more model seeds in the **seeds** box, comma-separated (e.g. `1,2,3`). Each seed is an independent prediction (more seeds = more sampling, more runtime). Non-numeric characters are ignored and duplicates are dropped, so `1, 1, foo, 7` becomes seeds `1` and `7`.

## MSA modes

- **`mmseqs2_server`** *(recommended)*: queries the public [ColabFold](https://colabfold.mmseqs.com/) MMseqs2 API. Covers UniRef30 + environmental sequences for proteins. RNA/DNA chains always run MSA-free (ColabFold is protein-only).
- **`single_sequence`**: no MSA, query sequence only. Faster but less accurate, especially for monomers with close homologs.

## Output files (inside the downloaded zip)

| File | Contents |
|---|---|
| `*.cif` | Best-ranked structure in mmCIF format. B-factor = pLDDT (0–100). |
| `*_confidences.json` | Per-residue pLDDT, PAE matrix, contact probs. |
| `*_summary_confidences.json` | Mean pLDDT, pTM, ipTM, ranking score. |
| `*_ranking_scores.csv` | Ranking scores for all seed × sample combinations. |
| `seed-N_sample-M/` | Individual prediction directories (one per seed/sample). |
| `TERMS_OF_USE.md` | Apache 2.0 notice (OpenFold3 weights have no commercial restrictions). |

## Interpreting confidence scores

- **pLDDT > 90**: very high confidence.
- **pLDDT 70–90**: confident, backbone generally reliable.
- **pLDDT 50–70**: low confidence, treat with caution.
- **pLDDT < 50**: very low, likely disordered or incorrect.
- **PAE**: lower values = confident relative positioning between residue pairs. Useful for assessing interface quality in complexes.
- **ipTM > 0.8**: strong evidence for a well-defined complex interface. **ipTM is `n/a` for single-chain jobs** (there is no interface to score).

## Troubleshooting

- **Check runtime type**: `Runtime → Change runtime type → GPU` (T4 is fine; A100/L4 are faster).
- **OOM error**: reduce sequence length or use a larger-memory GPU runtime.
- **MSA server timeout**: the public ColabFold server is rate-limited. Try again later or switch to `single_sequence` mode.
- **Download popup blocked**: disable your ad blocker.
- **Weight download slow**: the OF3 checkpoint is ~1.4 GB from AWS S3; the install cell waits automatically.

## License

AlphaFold 3 source code and OpenFold3 weights are both licensed under [Apache 2.0](https://www.apache.org/licenses/LICENSE-2.0). Outputs generated with OpenFold3 weights are **not** subject to Google DeepMind's AlphaFold 3 Output Terms of Use and may be used freely, including commercially.

## Bugs / feedback

Report issues at https://github.com/sokrypton/alphafold3/issues